# AcademicExtract-R1 · SFT 冒烟（Colab T4）
步骤：跑 Cell 1 装环境（约几分钟）→ 左侧文件区拖入 `colab_smoke_pack.zip` → 依次跑 Cell 2/3/4。
预期：20 步训练完成、loss 下降、产出 run_summary.json + LoRA。

In [ ]:
%%capture
# Unsloth 官方 pin 组合（Kaggle-Qwen3_(4B)-GRPO.ipynb cell 0 原味, T4 感知）
import os
!pip install --upgrade -qqq uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: is_t4 = False
_vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
!uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
!uv pip install -qqq {_triton} "huggingface_hub>=0.34.0" "datasets==4.3.0"
!uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [ ]:
# Cell 2: 解压冒烟包（先把 colab_smoke_pack.zip 拖进左侧文件区）
import zipfile, os
assert os.path.exists('colab_smoke_pack.zip'), '请先把 colab_smoke_pack.zip 拖入文件区'
zipfile.ZipFile('colab_smoke_pack.zip').extractall('.')
print(os.listdir('.'))

In [ ]:
# Cell 3: SFT 冒烟（40 条 + 20 步, T4 约十几分钟）
!python sft_train.py --smoke --data sft_train.jsonl --batch-size 1 --grad-accum 2

In [ ]:
# Cell 4: 打包产物带回（下载 sft_outputs.zip 发回）
import shutil
shutil.make_archive('sft_outputs', 'zip', 'outputs')
from google.colab import files
files.download('sft_outputs.zip')